In [ ]:
import json
import os
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import pandas as pd
import pysam
import pyfaidx
import pybedtools
import csv
import tensorflow as tf

from baskerville import dataset
from baskerville import seqnn
from baskerville import gene as bgene
from baskerville import dna
from baskerville import trainer

tf.compat.v1.logging.set_verbosity(tf.compat.v1.logging.ERROR)
#os.environ['CUDA_VISIBLE_DEVICES'] = '-1'


In [ ]:
#Evaluate bin-level performance on a subset of tracks

data_dir = 'data/v11s/mm10'
model_file = 'saved_models/f3c0/train/model1_best.h5'
params_file = 'params.json'
targets_file = 'targets_mouse.txt'
split_label = 'fold3'
head_i = 1
rc = True
shifts = [0]
batch_size = 1

#Targets into model head
model_target_index = [
    1560, #RNA3:CD4-positive, alpha-beta T cell
    1561, #RNA3:CD4-positive, alpha-beta T cell
    1720, #RNA3:naive B cell
    1721, #RNA3:naive B cell
    1762, #RNA3:skeletal muscle cell
    1763, #RNA3:skeletal muscle cell
]

num_targets = len(model_target_index)

#Read targets
targets_df = pd.read_csv(targets_file, index_col=0, sep="\t")

#Read model parameters
with open(params_file) as params_open:
    params = json.load(params_open)
params_model = params["model"]
params_train = params["train"]

#Set strand pairs (using new indexing)
orig_new_index = dict(zip(targets_df.index, np.arange(targets_df.shape[0])))
targets_strand_pair = np.array(
    [orig_new_index[ti] for ti in targets_df.strand_pair]
)
params_model["strand_pair"] = [targets_strand_pair]

#Construct eval data
eval_data = dataset.SeqDataset(
    data_dir,
    split_label=split_label,
    batch_size=batch_size,
    mode="eval",
    tfr_pattern=None,
)

#Initialize model
seqnn_model = seqnn.SeqNN(params_model)
seqnn_model.restore(model_file, head_i)
seqnn_model.build_slice(targets_df.index)
seqnn_model.build_ensemble(rc, shifts)

#Read sequence intervals

# read data parameters
with open("%s/statistics.json" % data_dir) as data_open:
    data_stats = json.load(data_open)
    crop_bp = data_stats["crop_bp"]
    pool_width = data_stats["pool_width"]

seqs_df = pd.read_csv(
    "%s/sequences.bed" % data_dir,
    sep="\t",
    names=["Chromosome", "Start", "End", "Name"],
)
seqs_df = seqs_df[seqs_df.Name == split_label]


In [5]:
#Predict test sequences
from baskerville import metrics

pearsonr_metric = metrics.PearsonR(num_targets, summarize=False)

n_seqs = None #Set to None to test on all sequences

#Loop over dataset rows
for i, [x, y] in enumerate(eval_data.dataset) :
    
    if i % 100 == 0 :
        print("Predicting for row " + str(i) + "...")
    
    if n_seqs is not None and i >= n_seqs :
        break
    
    #Get predictions and meaasurements
    y_pred = seqnn_model(x)[..., model_target_index]
    
    pearsonr_metric(y, tf.convert_to_tensor(y_pred, dtype='float32'))

#Calculate metrics
pearsonrs = pearsonr_metric.result().numpy()

print("pearsonrs = " + str(pearsonrs))


Predicting for row 0...


2026-09-14 12:06:55.303145: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2026-09-14 12:06:58.290638: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory
2026-09-14 12:06:59.302453: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Predicting for row 100...
Predicting for row 200...
Predicting for row 300...
Predicting for row 400...
Predicting for row 500...
Predicting for row 600...
Predicting for row 700...
Predicting for row 800...
Predicting for row 900...
Predicting for row 1000...
Predicting for row 1100...
Predicting for row 1200...
Predicting for row 1300...
Predicting for row 1400...
Predicting for row 1500...
Predicting for row 1600...
Predicting for row 1700...
Predicting for row 1800...
Predicting for row 1900...
Predicting for row 2000...
Predicting for row 2100...
Predicting for row 2200...
Predicting for row 2300...
Predicting for row 2400...
Predicting for row 2500...
Predicting for row 2600...
Predicting for row 2700...
Predicting for row 2800...
Predicting for row 2900...
Predicting for row 3000...
Predicting for row 3100...
Predicting for row 3200...
Predicting for row 3300...
Predicting for row 3400...
Predicting for row 3500...
Predicting for row 3600...
Predicting for row 3700...
Predicting